# Direction
1.Data preprocessing   
2.Connect <b>ContextBuilder</b> to <b>BERT</b>  get one set of benign sequences and benign anomaly sequences.   
3.Try to use self-supervision method to label.   
4.Use one of <b>Cybersecurity-LLMs</b>[Lily, CTI-BERT...to compare] to parse, decomposition detect and explain.   
5.To use <b>RAG</b> to improve detection performance.   
6.Option: RAG evaluation   
7.LLM evaluation   

<b>necessary folder description:</b> <br/>
PredataBERT: preprocessed benign normal data ( converted format and constructed data) <br/>
PredataBERT: preprocessed benign anomaly data.<br/>
saved_model: store args, ContextBuilder model, and Interpreter model.<br/>
checkpoints: store the SequenceDetector model (the latest and the best)<br/>
result: the result file of SequenceDetector( label for seq.), history experiment result of SequenceDetector, history expeirment result of Interpreter. 

<b>description for symbolic: </b><br/>
data plan 1 (100 hosts in one day)<br/>
filename-2 (for 'classic' mode)<br/>
filename-3 ( for  'ParamEncode' mode)<br/>

data plan 2 (5 hosts in one week)<br/>
filename-2-2   (for 'classic' mode)<br/>
filename-3-2    ( for  'ParamEncode' mode)<br/>

<b>workflow:<br/></b>
step1 for training ContextBuilder and sequenceDetector on all datafiles. AICMain.ipynb, for_allfiles=True.<br/>
    run form beginning section untill 'BERT' section.<br/>
    the trained ContextBilder and SequenceDetector are saved in specific folder in device.<br/>

step2 for producing event seq. label/levels on one specific datafile including the test datafile.     using AICMain.ipnyb, for_allfiles=False.<br/>
    run from beginning section , by loading existed ContextBilder and SequenceDetector.<br/>
    untill 'label whole file data' section.<br/>
    the score/label/level of this datafile is saved in specific folder in device.<br/>

step3 for testing using Interprer on all datafiles-manul/semi-automatic analysis.    AICMain.ipynb, for_allfiles=True.<br/>
    run from beginning section, untill 'DataPreprocessing'<br/>
    run the first section in 'ContextBuilder' section to load existed ContextBuilder.<br/>
    run 'Interpreter' on train dataset to get final result in manual analysis. <br/>
    run 'Interpreter' on test dataset to get final result in semi-automatic analysis. <br/>

# prepare GPUs

In [ ]:
import os

# nvidia-smi

# Limit PyTorch to use specific GPUs
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  #"2,3,4,5"  # 

import torch

torch.cuda.init()  # or a simple cuda call like:
# torch.cuda.current_device()

# Check available GPUs
print("Available GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_ids =[0,1,2,3]  # [2,3,4,5]  # used in BERT for parallel
print('Using device = ', device)
    


Available GPUs: 4
GPU 0: NVIDIA H100 80GB HBM3
GPU 1: NVIDIA H100 80GB HBM3
GPU 2: NVIDIA H100 80GB HBM3
GPU 3: NVIDIA H100 80GB HBM3
Using device =  cuda


In [2]:

import gc
gc.collect()
torch.cuda.empty_cache()

# Print memory allocated for each GPU
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Allocated: {torch.cuda.memory_allocated(i)/1024**2:.2f} MB")
    print(f"  Cached:    {torch.cuda.memory_reserved(i)/1024**2:.2f} MB")
    
print(torch.__version__)
if torch.cuda.is_available():
    print(torch.version.cuda) 

# torch.version=2.5.1
# cuda=12.1

GPU 0: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
GPU 1: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
GPU 2: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
GPU 3: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
2.5.1
12.1


In [3]:

import numpy as np
import sys
sys.path.append('..')

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import random
# from collections import Counter
import json

# DeepCASE Imports
from deepcase.preprocessing   import Preprocessor
from deepcase.context_builder import ContextBuilder
from deepcase.interpreter     import Interpreter


In [4]:
from AIC.utils import *

In [ ]:
# for BERT
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MinMaxScaler
import torch.nn.functional as F
from tqdm import tqdm
from model import Model
from log_attention import LogAttention
from dataloader import DataGenerator

import torch.nn as nn
import torch.optim as optim
import time
import tempfile
import gc

In [7]:
print(pd.__version__)  # 2.2.3
print(sys.version) # python
# torch version=2.5.1
# python=3.10.16  ， 3.10.18
# cuda=12.1


2.2.3
3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]


# temp for extract the data from one host in one day

In [ ]:
file_path="../OpTC-part/BenignAnomalyHost-csv/18-19-123.csv"

df = pd.read_csv(file_path, parse_dates=["timestamp"])
# filtered = df[df["timestamp"].dt.date == pd.to_datetime("2019-09-20").date()]
# filtered.to_csv("../OpTC-part/BenignAnomalyHost-csv/20-23-124(20).csv", index=False)   

In [ ]:
counts_by_day = df["timestamp"].dt.date.value_counts().sort_index()

# Print each day with its count
for day, count in counts_by_day.items():
    print(f"{day}: {count} records")

In [ ]:
def label_csv_with_json(csv_file, json_file):
    # Load JSON file and collect all "id" values into a set
    json_ids = set()
    with open(json_file, "r", encoding="utf-8") as f_json:
        for line in f_json:
            try:
                record = json.loads(line)
                if "id" in record:
                    json_ids.add(record["id"])
            except json.JSONDecodeError:
                continue

    # Load the CSV file
    df = pd.read_csv(csv_file)

    # Label: 1 if 'id' exists in JSON, else 0
    df["label"] = df["id"].apply(lambda x: 1 if x in json_ids else 0)

    # Save the labeled CSV to a new file
    output_file = os.path.splitext(csv_file)[0] + "_labeled.csv"
    df.to_csv(output_file, index=False)
    print(f"Labeled file saved as: {output_file}")

In [ ]:
label_csv_with_json("../OpTC-part/BenignAnomalyHost-csv/20-23-124(20).csv","../OpTC-part/admin.json")


In [ ]:
def has_positive_labels(csv_file):
    df = pd.read_csv(csv_file)
    
    if "label" not in df.columns:
        print("No 'label' column found in the file.")
        return

    count_1 = (df["label"] == 1).sum()
    
    print(f"Records with label 1: {count_1}")

In [ ]:
has_positive_labels("../OpTC-part/BenignAnomalyHost-csv/20-23-124(20)_labeled.csv")

# reconstruct data if it is raw from OpTC

In [ ]:
'''convert json to csv & align timestamp'''

# 18-23 Sep
# input_folder = "../OpTC-part/ecar-bro-benign-hosts-5days"  # json files
# output_folder = "../OpTC-part/ecar-bro-benign-hosts-5days-csv"
# 19 Sep
# input_folder = "../OpTC-part/ecar-bro-benign-hosts-19Sep19"  # json files
# output_folder = "../OpTC-part/ecar-bro-benign-hosts-19Sep19-csv"

# new files
input_folder = "../OpTC-part/BenignAnomalyHost"  # json files
output_folder = "../OpTC-part/BenignAnomalyHost-csv"

# Loop through all JSON files in the input folder
json2csv_timestamp(input_folder,output_folder)

In [ ]:
'''assign fields and values for BERT'''

# file_folder="../OpTC-part/ecar-bro-benign-hosts-5days-csv"
# file_folder="../OpTC-part/ecar-bro-benign-hosts-19Sep19-csv"
file_folder="../OpTC-part/BenignAnomalyHost-csv"

# extract_folder='./PredataBERT'
extract_folder='./PreAnomalydataBERT'

short_file='shortBERT.csv'

reConstructData( file_folder,extract_folder,short_file)


In [ ]:
''' save event index and event type name in 'shortBERTIndex.csv' '''
readname='shortBERT.csv'
addIndex4Uniq(readname)

In [ ]:
from datetime import datetime

# Assuming your dataframe is called 'df' and contains 'timestamp' and 'hostname' columns

# Convert timestamp to datetime if it's not already
if not pd.api.types.is_datetime64_any_dtype(labelset['timestamp']):
    labelset['timestamp'] = pd.to_datetime(labelset['timestamp'],format='mixed')

# Extract just the date part (without time)
labelset['date'] = labelset['timestamp'].dt.date

# Group by date and hostname, then count
result = labelset.groupby(['date', 'hostname']).size().reset_index(name='count')

# Sort by date and then by count (descending) if desired
result = result.sort_values(['date', 'count'], ascending=[True, False])

print(result)

# params prepare

In [6]:
import argparse
from argparse import Namespace

def save_args(args, path="saved_model/args.json"):
    """Save argparse.Namespace to a JSON file"""
    with open(path, "w") as f:
        json.dump(vars(args), f, indent=2)
        
def load_args(path="saved_model/args.json"):
    """Load argparse.Namespace from a JSON file"""
    with open(path, "r") as f:
        args_dict = json.load(f)
    return Namespace(**args_dict)

In [ ]:

args=load_args()

In [7]:

args=Namespace(
    # mode=   'classifier', # sequence detector without log parameters encoder using Transformer with multiheadattenion
    mode=  'paramEncoder', # concern log parameters encoder in Transformer instead of multiheadattention
    events='auto',  #help="number of distinct events to handle"
    length=10, # help="sequence LENGTH, window size")
    timeout=86400, #help="sequence TIMEOUT (seconds)")
    # save_sequences=None,  # help="path to save sequences")
    # load_sequences=None, # help="path to load sequences")
    inital_param_size=14605356,
########################### Add ContextBuilder arguments
    hidden=128, # help="HIDDEN layers dimension")
    # delta=0.1,  # help="label smoothing DELTA") fixed in source code of CB, no need this.
    save_builder='saved_model/saved_CB_model.pth',  #help="path to save ContextBuilder")
    load_builder=None, # help="path to load ContextBuilder")
    
######################## Add Training arguments for ContextBuilder
    epochs_CB=  10, # 50 , # help="number of epochs to train with")
    batch_CB= 4096, # 128, #help="batch size       to train with")
    learning_rate_CB=0.001,
####################### Add Interpreter arguments
    confidence=0.1,  # help="minimum required CONFIDENCE", 0.2 in paper
    epsilon=0.01,  # help="DBSCAN clustering EPSILON", 0.1 in paper
    min_samples=5  , # help="DBSCAN clustering MIN_SAMPLES", 5 in paper
    save_interpreter='saved_model/saved_Interpreter_model.pth', #  help="path to save Interpreter")
    load_interpreter=None, #  help="path to load Interpreter")
# group_interpreter.add_argument('--save-clusters'   , help="path to CSV file to save clusters")
# group_interpreter.add_argument('--load-clusters'   , help="path to CSV file to load clusters")
    save_prediction=None, # help="path to CSV file to save prediction")
    cluster_metric="labels",  # the metric used to calculate the score of clusters.

#################### Add SequenceDetector arguments
    resume=0, #help="resume training of model (0/no, 1/yes)")
    load_SequenceDetecter='checkpoints/model-latest.pt', # help="latest model path")

#################### Add Training arguments for Sequence Detector
    epochs_SD=20, # 50 , # help="number of epochs to train with")
    batch_SD=12*10, # help="batch size to train with, **** must be****  divided by window size.")
    lr_SD=0.0001,  
    num_layers_SD=1, # help='num of encoder layer')
    threshold_SD=0.5, # help='threhold value for evaluation') decided after training.
    embed_content_dim=768,
    embed_param_dim=3,
    embed_context_dim=128,
    alpha_loss=0.5,
    
#################### Add DataLoader arguments
    dataloader_batch_size=1280,
    dataloader_chunk_size=400000,

####################### Add other arguments
    device='auto' , #  help="DEVICE used for computation (cpu|cuda|auto)")
    silent=True, # action='store_true', help="silence mode, do not print progress")

# useless. use when concern file split
    train_test_size=0.1,   # help="split all data into training set and test set")
    train_validate_size=0.2,  # help="split training set into train set and validate set")

)

# DataPreprocessing
file 20-23-123(20).csv as validate data in SD.
20-23-124(20).csv as test data .

In [8]:
########################################################################
#                             Loading data                             #
########################################################################

# Create preprocessor
preprocessor = Preprocessor(
    length  = args.length,    # 10 events in context, also window size in Sequence Detector
    timeout = args.timeout,  #  86400, # Ignore events older than 1 day (60*60*24 = 86400 seconds)
)
    

In [ ]:
# load data from files
file_folder = "PredataBERT"

for_allfiles= True # False  # flag if process all csv files or just one file

if for_allfiles:
    for_interpreter= True #False # flag whether need tha score/label/level of train data.

    # Process all CSV files in the folder at one time
    frame_list = []
    scores_list=[]
    labels_list=[]
    levels_list=[]
    cumulative_info = []  # List of tuples (file_name, cumulative_length)
    cum_len = 0

    for file_name in os.listdir(file_folder):    
        # if file_name.endswith(".csv"): # visit all csv file
        if file_name.startswith("AIA-"):    # visit 100 hosts in one day
            file_path = os.path.join(file_folder, file_name)  
            data_temp=pd.read_csv(file_path, usecols=['id','timestamp','hostname','event','field_names','field_values'])[['id','timestamp','hostname','event','field_names','field_values']]
            # ############ temperaly get 10000 records
            frame_list.append(data_temp)   # .loc[:1000000,:]) 
            print("read file:"+ file_path+" , len="+ str(len(data_temp)) )

            cum_len += len(data_temp)
            cumulative_info.append((file_path, cum_len))

            if for_interpreter:
                label_filename="result/"+ file_name.replace(".csv", "-onlyScoreLabel.csv")
                scores_temp, labels_temp, levels_temp = load_anomaly_results(label_filename)
                scores_list.append(pd.DataFrame(scores_temp) )
                labels_list.append( pd.DataFrame(labels_temp)  )
                levels_list.append( pd.DataFrame(levels_temp)  )

    data = pd.concat(frame_list)

    if for_interpreter:
        final_scores= pd.concat(scores_list).to_numpy().squeeze(1)
        final_labels=pd.concat(labels_list).to_numpy().squeeze(1)
        final_levels=pd.concat(levels_list).to_numpy().squeeze(1)

    data.reset_index(drop=True, inplace=True)

    del data_temp
    del frame_list
    torch.cuda.empty_cache() 

else:
    
    test_filename = "AIA-51-75.ecar-last.csv" 

    file_path= os.path.join(file_folder, test_filename) 
     
    data=pd.read_csv(file_path, usecols=['id','timestamp','hostname','event','field_names',
                                     'field_values']   )[['id','timestamp','hostname','event','field_names'
                                                       ,'field_values']]

data = data.rename(columns = {"hostname":"machine"})
data['label']=0 # label for each event. not work 


In [ ]:
''' preprocess for ContextBuilder'''

mapping=get_event_mapping("shortBERTIndex.csv")

start_time = time.time()
context_train, events_train, mapping_label, labels_train_binary=preprocessor.sequence(data,labels=None,verbose=False, existed_mapping=mapping)
time_cost = time.time()-start_time
print(f"Time cost: {time_cost:.3f} seconds")

# print(f'the total number of samples: {len(labels)}')
# print(f'the number of false positive: {sum(labels==0)}')
# print(f'the number of analomal samples: {sum(labels==1)} ')


# Automatically set device argument
if args.device == "auto":
    args.device = "cuda" if torch.cuda.is_available() else "cpu"
    # args.device="cpu" # v100

# Automatically set the number of events to expect
if args.events == "auto":
    args.events = len(mapping) +1
else:
    args.events = int(args.events)

In [ ]:
# Cast tensors to device
events_train  = events_train.to(args.device)
context_train = context_train.to(args.device)

## save& load preprocessed data

In [ ]:
# temperaly save data in device in need
if for_allfiles:    
    data.to_csv("saved_model/alldata100.csv",index=False)

    torch.save(context_train, "checkpoints/context_train.pt ")
    torch.save( events_train, "checkpoints/events_train.pt ")
    torch.save( mapping_label, "checkpoints/mapping_label.pt ")
    torch.save( labels_train_binary , "checkpoints/labels_train_binary.pt ")

    output_filename= "saved_model/alldata100-onlyScoreLabel.csv"
    save_results_to_csv(final_scores, final_labels,final_levels, output_filename)

else:

    torch.save(context_train, "checkpoints/context_train_51_75.pt")
    torch.save( events_train, "checkpoints/events_train_51_75.pt")
    torch.save( mapping_label, "checkpoints/mapping_label_51_75.pt")
    torch.save( labels_train_binary , "checkpoints/labels_train_binary_51_75.pt")
    torch.save( data , "checkpoints/data_51_75.pt")
    
    # torch.save(context_train, "checkpoints/context_train_18_51.pt")
    # torch.save( events_train, "checkpoints/events_train_18_51.pt")
    # torch.save( mapping_label, "checkpoints/mapping_label_18_51.pt")
    # torch.save( labels_train_binary , "checkpoints/labels_train_binary_18_51.pt")
    
    print("prprocessed data saved in checkpoints.")


### load

In [ ]:
# only when need to load all data from existed csv.
for_allfiles= True  # False #
if for_allfiles:   
    # data=pd.read_csv("saved_model/alldata100.csv")
    # print(data.head())

    test_filename = "alldata100.csv"

    context_train=torch.load("checkpoints/context_train.pt ")
    events_train=torch.load("checkpoints/events_train.pt ") 
    mapping_label=torch.load("checkpoints/mapping_label.pt ") 
    labels_train_binary=torch.load("checkpoints/labels_train_binary.pt ")

    label_filename="saved_model/alldata100-onlyScoreLabel.csv"
    final_scores, final_labels, final_levels = load_anomaly_results(label_filename)


else:
    test_filename = "AIA-51-75.ecar-last.csv"
    
    data=torch.load("checkpoints/data_51_75.pt") 
    context_train=torch.load("checkpoints/context_train_51_75.pt")
    events_train=torch.load("checkpoints/events_train_51_75.pt") 
    mapping_label=torch.load("checkpoints/mapping_label_51_75.pt") 
    labels_train_binary=torch.load("checkpoints/labels_train_binary_51_75.pt")
    


In [ ]:
# load data from files
file_folder = "PredataBERT"


mapping=get_event_mapping("shortBERTIndex.csv")
# Automatically set device argument
if args.device == "auto":
    args.device = "cuda" if torch.cuda.is_available() else "cpu"
    # args.device="cpu" # v100

# Automatically set the number of events to expect
if args.events == "auto":
    args.events = len(mapping) +1
else:
    args.events = int(args.events)  # +1

In [11]:
# Cast tensors to device
events_train  = events_train.to(args.device)
context_train = context_train.to(args.device)

## specific validate file data

In [31]:
test_file ="PreAnomalydataBERT/20-23-123(20).csv"
label_file=("BenignAnomalyHost-csv/20-23-123(20)_labeled.csv")
            
data_val=pd.read_csv(test_file, usecols=['id','timestamp','hostname','event','field_names','field_values'])[['id','timestamp','hostname','event','field_names','field_values']]
data_val = data_val.rename(columns = {"hostname":"machine"})

data_val['label']=pd.read_csv(label_file, usecols=["label"]) 

In [17]:
start_time = time.time()
context_val, events_val, _, labels_val_binary=preprocessor.sequence(data_val,labels=None,verbose=False, existed_mapping=mapping)
time_cost = time.time()-start_time
print(f"Time cost: {time_cost:.3f} seconds")

# print(f'the total number of samples: {len(labels_val_binary)}')
# print(f'the number of false positive: {sum(labels_val_binary==0)}')
# print(f'the number of analomal samples: {sum(labels_val_binary==1)} ')

Time cost: 158.010 seconds


In [18]:

frames=[pd.DataFrame(context_val.detach().numpy()), data_val]  # 
X_val = pd.concat(frames, axis=1)  

frames=[pd.DataFrame(context_train.cpu().detach().numpy()), data]  # 
X_train = pd.concat(frames, axis=1)  

# del data_val, data

In [19]:
# Cast tensors to device
events_val  = events_val.to(args.device)
context_val = context_val.to(args.device)

## specific test file data

In [ ]:

test_file ="PreAnomalydataBERT/20-23-124(20).csv"
label_file=("BenignAnomalyHost-csv/20-23-124(20)_labeled.csv")
            
data_test=pd.read_csv(test_file, usecols=['id','timestamp','hostname','event','field_names','field_values'])[['id','timestamp','hostname','event','field_names','field_values']]
data_test = data_test.rename(columns = {"hostname":"machine"})

data_test['label']=pd.read_csv(label_file, usecols=["label"]) 


In [ ]:
start_time = time.time()
context_test, events_test, _, labels_test_binary=preprocessor.sequence(data_test,labels=None,verbose=False, existed_mapping=mapping)
time_cost = time.time()-start_time
print(time_cost)

print(f'the total number of samples: {len(labels_test_binary)}')
print(f'the number of false positive: {sum(labels_test_binary==0)}')
print(f'the number of analomal samples: {sum(labels_test_binary==1)} ')

In [ ]:


frames=[pd.DataFrame(context_test.detach().numpy()), data_test]  # 
X_test = pd.concat(frames, axis=1)  


In [ ]:
# Cast tensors to device
events_test  = events_test.to(args.device)
context_test = context_test.to(args.device)

# ContextBuilder

In [ ]:
########################################################################
#                         Using ContextBuilder                         #
########################################################################

# args.load_builder=  None   #
args.load_builder= "saved_model/saved_CB_model-ori-128_Adamx_100.pth"
# Load the builder, if necessary
if args.load_builder:
    context_builder = ContextBuilder.load(args.load_builder, args.device)

# Otherwise create a new ContextBuilder
else:
    print( "origin DeepCASE for testing")
    # Create ContextBuilder
    context_builder = ContextBuilder(
        input_size    = args.events,
        output_size   = args.events,
        hidden_size   = 128, # args.hidden, 128  # in paper from 2 -1024
        num_layers    = 1,
        max_length    = args.length,
        bidirectional = False, 
        LSTM          = False,  
    ).to(args.device)


In [ ]:

args.epochs_CB=  50
args.batch_CB=1024 # 2048  
args.learning_rate_CB= 0.001 

if len(events_train)>args.dataloader_chunk_size :  # * 3:    
    context_builder=process_large_data_CB_fit(context_train, events_train,labels_train_binary,args,context_builder)
    
else:       
    # Train the ContextBuilder
    context_builder.fit(
        X             = context_train,               # Context to train with
        y             = events_train.reshape(-1, 1), # Events to train with, note that these should be of shape=(n_events, 1)
        labels        = labels_train_binary,
        epochs        = args.epochs_CB,                          # Number of epochs to train with
        batch_size    = args.batch_CB,                         # Number of samples in each training batch, in paper this was 128
        learning_rate = args.learning_rate_CB,                        # Learning rate to train with, in paper this was 0.01
        verbose       =  args.silent, #  not                       # If True, prints progress
        optimizer= optim.Adamax,  # optim.SGD,
        teach_ratio=0.5
    )

    


In [ ]:
# args.save_builder=None 
args.save_builder="saved_model/saved_CB_model-ori-128_Adamx_single.pth"
###### Save the builder, if necessary
if args.save_builder:
    context_builder.save(args.save_builder)

In [ ]:
### Debugging part, Test the accuracy of the context builder. where condidence value choice.
# for tune CB: 
if  True: #DEBUG:
    confidence_train, _ =process_large_data_CB_predict(context_train, events_train,args, context_builder)
    confidence_train=confidence_train.squeeze(1).exp()
    # predicted_val    = torch.argmax(confidence_val,dim=1).to('cpu').numpy()
    c_train, predicted_train = torch.max(confidence_train, dim=1)
    c_train=c_train.to('cpu').numpy()
    predicted_train=predicted_train.to('cpu').numpy()

    # evaluate on all data
    events_train_np = events_train.to('cpu').numpy()
    print(classification_report(events_train_np, predicted_train,digits=4))

    # evaluate on the data above confidence threshold
    performance_CB, condidence_threshold=threshold_search_confidence(events_train_np,c_train, predicted_train ,True)
    confidence_mask=c_train>condidence_threshold
    
    with open("result/CB_metric_output.txt", "a") as f:
        print("evaluate on train set:", file=f)
        print(datetime.fromtimestamp(time.time()),file=f)
        print(classification_report(events_train_np, predicted_train,digits=4),file=f)
        print(f"the best condidence threshold is {condidence_threshold}",file=f)

# if best:
    args.confidence=condidence_threshold

## evaluate CB on validate data

In [ ]:
if len(events_val)>args.dataloader_chunk_size * 3:
    context_builder=process_large_data_CB_fit(context_val, events_val,labels_val_binary,args,context_builder)
else:    
    context_builder.fit(
        X             = context_val,               # Context to train with
        y             = events_val.reshape(-1, 1), # Events to train with, note that these should be of shape=(n_events, 1)
        labels        = labels_val_binary,
        epochs        = args.epochs_CB,                          # Number of epochs to train with
        batch_size    = args.batch_CB,                         # Number of samples in each training batch, in paper this was 128
        learning_rate = 0.001,                        # Learning rate to train with, in paper this was 0.01
        verbose       =  args.silent,  # not                      # If True, prints progress
        optimizer=  optim.Adamax,  # optim.SGD,
        teach_ratio=0.5
    )

In [ ]:
### Debugging part, Test the accuracy of the context builder. where condidence value choice.

if  True: #DEBUG:
    confidence_val, _ =process_large_data_CB_predict(context_val, events_val,args, context_builder)
    confidence_val=confidence_val.squeeze(1).exp()
    c_val, predicted_val = torch.max(confidence_val, dim=1)
    c_val=c_val.to('cpu').numpy()
    predicted_val=predicted_val.to('cpu').numpy()

    events_val_np = events_val.to('cpu').numpy()
    print(classification_report(events_val_np, predicted_val,digits=4))

    with open("result/CB_metric_output.txt", "a") as f:
        print("evaluate on test set:", file=f)
        print(time.time(),file=f)
        print(classification_report(events_val_np, predicted_val,digits=4),file=f)
        print(f"the best condidence threshold is {c_val.min()}",file=f)


In [ ]:
# del data
torch.cuda.empty_cache()  # Clears unused memory
torch.cuda.ipc_collect()  # Helps reclaim fragmented memory


# BERT
<!-- ✅ Benefits of feeding pre-computed embedding vectors:
Modularity / Flexibility:
You can experiment with different embedding strategies (e.g., static word2vec, contextual BERT, handcrafted encodings) without changing your main model.
You could plug in learned embeddings from another pretrained model.

Better control:
You know exactly what the input looks like (embedding size, structure, etc.).
You can normalize, mask, augment, or mix embeddings in custom ways.

Lower parameter count:
If you're not learning an embedding layer, your model has fewer trainable parameters — useful when data is limited or you're freezing upstream models.

Fine-tuned representations:
If embeddings are pretrained or learned from a different task, they might encode semantic structure that improves generalization. -->

In [ ]:
from sentence_transformers import SentenceTransformer
embmodel = SentenceTransformer('distilbert-base-nli-mean-tokens', device=args.device)

/home/tengfei/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True

In [28]:

# get param_v_size for logattention in Model. use maximum param_size on whole dataset by default.
existed_param_size=False # True # 
if not existed_param_size:    
    f_v=pd.concat([X_train['field_values'], X_val['field_values']], axis=0)  
    unique_values =f_v.unique()  # Get unique values
    # one example: 0, PING.EXE, inbound, 17.0, 127.0.0.1, 64290.0
    # Save char_vocab
    pd.DataFrame(unique_values).to_csv("saved_model/"+test_filename+"_char_vocab.json", index=False)    
    param_v_size=len(unique_values)

    del f_v, unique_values
else:
    # Load previous vocab
    old_vocab=pd.read_csv("saved_model/all_2_char_vocab.csv")
    param_v_size=len(old_vocab)
    
print(param_v_size)
if args.inital_param_size==0:
    args.inital_param_size=param_v_size


4197840


In [ ]:
''' in the case of update parameters embedding:
1.Load Existing Vocab and Check for New Characters.
2.If New Characters Exist: Expand Vocab and Embedding Layer. 
3.Save the Updated Vocab for Future Use.
not test yet.
'''
if args.inital_param_size<param_v_size:
    # Load previous vocab
    old_vocab =pd.read_csv("saved_model/all_2_char_vocab.csv")
    
    # Detect new characters
    new_chars = set(itme for line in new_vocab) - set(old_vocab.keys())
    
    # Create updated vocab
    updated_vocab = dict(old_vocab)
    next_index = max(updated_vocab.values()) + 1
    for char in sorted(new_chars):
        updated_vocab[char] = next_index
        next_index += 1
    
    updated_vocab.to_csv("saved_model/all_2_char_vocab.csv", index=False)
        


In [ ]:
########### to prepare train data
_, attention_train = process_large_data_CB_predict(context_train, events_train,args, context_builder)
attention_train=attention_train.squeeze(1)

########### to prepare test data 
_, attention_test = process_large_data_CB_predict(context_val, events_val,args, context_builder) 
attention_test=attention_test.squeeze(1)

Processing chunk 1/217 (indices 0 to 133333)
Processing chunk 2/217 (indices 133333 to 266666)
Processing chunk 3/217 (indices 266666 to 399999)
Processing chunk 4/217 (indices 399999 to 533332)
Processing chunk 5/217 (indices 533332 to 666665)
Processing chunk 6/217 (indices 666665 to 799998)
Processing chunk 7/217 (indices 799998 to 933331)
Processing chunk 8/217 (indices 933331 to 1066664)
Processing chunk 9/217 (indices 1066664 to 1199997)
Processing chunk 10/217 (indices 1199997 to 1333330)
Processing chunk 11/217 (indices 1333330 to 1466663)
Processing chunk 12/217 (indices 1466663 to 1599996)
Processing chunk 13/217 (indices 1599996 to 1733329)
Processing chunk 14/217 (indices 1733329 to 1866662)
Processing chunk 15/217 (indices 1866662 to 1999995)
Processing chunk 16/217 (indices 1999995 to 2133328)
Processing chunk 17/217 (indices 2133328 to 2266661)
Processing chunk 18/217 (indices 2266661 to 2399994)
Processing chunk 19/217 (indices 2399994 to 2533327)
Processing chunk 20/21

/tmp/ipykernel_758379/989100989.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  all_result1 = [torch.load(p) for p in result1_paths]
/tmp/ipykernel_758379/989100989.py:

Processing chunk 1/26 (indices 0 to 133333)
Processing chunk 2/26 (indices 133333 to 266666)
Processing chunk 3/26 (indices 266666 to 399999)
Processing chunk 4/26 (indices 399999 to 533332)
Processing chunk 5/26 (indices 533332 to 666665)
Processing chunk 6/26 (indices 666665 to 799998)
Processing chunk 7/26 (indices 799998 to 933331)
Processing chunk 8/26 (indices 933331 to 1066664)
Processing chunk 9/26 (indices 1066664 to 1199997)
Processing chunk 10/26 (indices 1199997 to 1333330)
Processing chunk 11/26 (indices 1333330 to 1466663)
Processing chunk 12/26 (indices 1466663 to 1599996)
Processing chunk 13/26 (indices 1599996 to 1733329)
Processing chunk 14/26 (indices 1733329 to 1866662)
Processing chunk 15/26 (indices 1866662 to 1999995)
Processing chunk 16/26 (indices 1999995 to 2133328)
Processing chunk 17/26 (indices 2133328 to 2266661)
Processing chunk 18/26 (indices 2266661 to 2399994)
Processing chunk 19/26 (indices 2399994 to 2533327)
Processing chunk 20/26 (indices 2533327 t

In [ ]:
print('vector embedding...')
# calculate vectors for all known distinct templates
########################## embed field names ####################################################
# ef_train=data['event'].str.extract(r'^([^_]*)_[^_]*').squeeze(1) + "_" +  data['field_names']
# ef_val=data_val['event'].str.extract(r'^([^_]*)_[^_]*').squeeze(1) + "_" +  data_val['field_names']
# all_fieldnames=pd.concat([ef_train,ef_val], axis=0)
# event_template= pd.DataFrame( {'EventTemplate': all_fieldnames.drop_duplicates(ignore_index=True)})

# fieldname_em_train,fieldva_ind_train=field_name_value_prepare(origdata_fieldname=ef_train,  
#     origdata_fieldvalue=X_train['field_values'] , embmodel=embmodel, event_template=event_template)
# fieldname_em_test,fieldva_ind_test=field_name_value_prepare(origdata_fieldname=ef_val,
#     origdata_fieldvalue=X_val['field_values'] , embmodel=embmodel, event_template=event_template)


# # # option2 more event feature name +part values         ## +field_names ########## must read orgin data again.
# ef_train=data['event']       + "_" +  data['field_names']  
# ef_val=data_val['event']       + "_" +  data_val['field_names']
# all_fieldnames=pd.concat([ef_train,ef_val], axis=0)
# event_template= pd.DataFrame( {'EventTemplate': all_fieldnames.drop_duplicates(ignore_index=True)})

# fieldname_em_train,fieldva_ind_train=field_name_value_prepare(origdata_fieldname=ef_train,  
#     origdata_fieldvalue=X_train['field_values']  , embmodel=embmodel, event_template=event_template)
# fieldname_em_test,fieldva_ind_test=field_name_value_prepare(origdata_fieldname=ef_val, 
#     origdata_fieldvalue=X_val['field_values'] , embmodel=embmodel, event_template=event_template)


# option3 field_names
all_fieldnames=pd.concat([X_train['field_names'], X_val['field_names']], axis=0)
event_template= pd.DataFrame( {'EventTemplate': all_fieldnames.drop_duplicates(ignore_index=True)})

fieldname_em_train,fieldva_ind_train=field_name_value_prepare(origdata_fieldname= X_train['field_names'], 
                                                              origdata_fieldvalue=X_train['field_values'] , embmodel=embmodel, event_template=event_template)
fieldname_em_test,fieldva_ind_test=field_name_value_prepare(origdata_fieldname= X_val['field_names'], 
                                                            origdata_fieldvalue=X_val['field_values'] ,embmodel=embmodel, event_template=event_template)


vector embedding...


In [ ]:
attention_train=attention_train.to(fieldname_em_train.device)
context_train=context_train.to(fieldname_em_train.device)

# combination of field name, field value, attention vector 
train_bert=torch.cat([fieldname_em_train,fieldva_ind_train,attention_train,context_train],dim=1)
print(train_bert.size())

#####################################################
attention_test=attention_test.to(fieldname_em_test.device)
context_val=context_val.to(fieldname_em_test.device)

# combination of field name, field value, attention vector 
test_bert=torch.cat([fieldname_em_test,fieldva_ind_test,attention_test, context_val],dim=1)
print(test_bert.size())
    
'''

## train transfomer model

In [ ]:
transformermodel = Model(mode=args.mode, num_layers=args.num_layers_SD, dim=args.embed_content_dim, window_size=args.length, nhead=8, dim_feedforward=4 *
              args.embed_content_dim, dim_ff_reduced=128, dropout=0.3,eventID_size=args.events,eventID_embed_dim=args.embed_context_dim, param_vocab_size=args.inital_param_size, param_embed_dim=args.embed_param_dim)

transformermodel = transformermodel.to(device)
transformermodel = torch.nn.DataParallel(transformermodel, device_ids=device_ids)


/home/tengfei/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  warnings.warn(


In [ ]:
# calculate the parameters size of model
# print(  sum(p.numel() for p in transformermodel.parameters())
params_size=  sum(p.numel() for p in transformermodel.parameters()) 

In [ ]:
# args.load_path='checkpoints/train_bert_latest-3.pt'
args.load_path='checkpoints/train_bert_best-SGDOCLR-8731.pt'
# args.load_path= None
path_checkpoint = args.load_path
checkpoint = torch.load(path_checkpoint)
filtered_dict = {k: v for k, v in checkpoint['net'].items() if k in transformermodel.state_dict()}
transformermodel.load_state_dict(filtered_dict, strict=False)

if args.inital_param_size<param_v_size:
    args.inital_param_size=param_v_size
    print(f"inital_param_size change to {param_v_size}")
    transformermodel.module.update_param_vocab(param_v_size)

In [ ]:
args.epochs_SD=  20
args.batch_SD=128  *10   *4
args.lr_SD=0.01 
alpha=args.alpha_loss

In [ ]:
''' note: test data for transformer model here acturally is validate data'''
train_generator = DataGenerator( train_bert.clone(), labels_train_binary.clone())  
train_loader = torch.utils.data.DataLoader(
    train_generator, batch_size=args.batch_SD, shuffle=True) 

test_generator = DataGenerator(test_bert.clone(), labels_val_binary.clone())  
test_loader = torch.utils.data.DataLoader(
    test_generator, batch_size=args.batch_SD, shuffle=False) 

In [31]:
del train_bert, labels_train_binary
del test_bert, labels_val_binary
gc.collect()
torch.cuda.empty_cache()

In [ ]:

optimizer = optim.SGD(transformermodel.parameters(), lr=args.lr_SD, momentum=0.9,weight_decay=1e-2)

# optimizer = optim.Adam(transformermodel.parameters(), lr=args.lr_SD, weight_decay=0)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='min', factor=0.7, patience=4, threshold=1e-4, verbose=True)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=args.lr_SD, epochs=args.epochs_SD, steps_per_epoch=len(train_loader))

start_epoch = -1


### train & test

In [ ]:
    

final_auc=0
log_interval =100
ckpt_path = args.load_SequenceDetecter.split('/')[0] 

for epoch in range(start_epoch+1, args.epochs_SD):
    loss_all= [] 
    train_loss = 0
    context_loss_list=[]
    content_loss_list=[]

    transformermodel.train()
    start_time = time.time()
    for batch_idx, (xx,y) in enumerate(tqdm(train_loader)):
        content_true=xx[:,:args.embed_content_dim]
        context_true=xx[:,-args.length:].long()
        x=xx[:,:-args.length]
        x = x.to(device, dtype=torch.float32)
        # y = y.to(device, dtype=torch.float32)
        recon_x, recon_context = transformermodel(x)
        recon_x=recon_x.cpu()
        recon_context=recon_context.cpu()
        
        content_loss = F.mse_loss(recon_x, content_true)
        context_loss = F.cross_entropy(recon_context.reshape(-1, recon_context.size(-1)), context_true.reshape(-1), reduction='mean')
        loss= alpha* context_loss+ (1-alpha) *content_loss
        
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(transformermodel.parameters(), 0.5)
        optimizer.step()
        train_loss += loss.item()

        if batch_idx % log_interval == 0 and batch_idx > 0:
            cur_loss = train_loss / log_interval
            time_cost = time.time()-start_time
            print(f'| epoch {epoch:3d} | {batch_idx:5d}/{len(train_loader):5d} batches | '
                  f'loss {cur_loss} |'
                  f'lr {scheduler.get_last_lr()}')

            loss_all.append(train_loss)
            context_loss_list.append(context_loss.detach().numpy())
            content_loss_list.append(content_loss.detach().numpy())
            
            start_time = time.time()
            train_loss = 0

    
    train_loss = sum(loss_all) / len(train_loader)
    print("epoch : {}/{}, loss = {:.6f}".format(epoch, args.epochs_SD, train_loss))
    
    scheduler.step()    #OneCycleLR
    # scheduler.step(train_loss)  # ReduceLROnPlateau 
    
    checkpoint = {
        "net": transformermodel.state_dict(),
        'optimizer': optimizer.state_dict(),
        "epoch": epoch
    }
    torch.save(checkpoint, os.path.join(
        ckpt_path, f'train_bert_latest-3.pt'))
    
    plt.figure(figsize=(6, 4))
    plt.scatter(range(0,len(context_loss_list)), context_loss_list, label='context_loss_list') 
    plt.scatter(range(0,len(content_loss_list)), content_loss_list, label='content_loss_list')  #, linewidth=1)
    plt.xlabel('size')
    plt.ylabel('loss')
    plt.legend()
    plt.grid(True)
    f=pyplot.gcf()
    f.savefig(attackname+'losscompare.pdf')
    f.clear()
    plt.show()

###################################### validating ###########################
    transformermodel.eval()
    y_true=[]
    en_list, ex_list, re_en_list, re_ex_list=[],[],[],[]
    with torch.no_grad():
        for batch_idx, (xx,y) in enumerate(tqdm(test_loader)): 
            content_true=xx[:,:args.embed_content_dim]    
            context_true=xx[:,-args.length:].long()
            x=xx[:,:-args.length]
            
            x = x.to(device, dtype=torch.float32)
            # y = y.to(device, dtype=torch.float32)

            recon_x, recon_context = transformermodel(x)  #.cpu()
        
            # Accumulate raw tensors first
            # y_pred.append(out)
            y_true.append(y)
            en_list.append(content_true)
            ex_list.append(context_true)
            re_en_list.append(recon_x)
            re_ex_list.append(recon_context)
            
    # After loop: stack once, move to CPU, convert to NumPy
    re_en_list = torch.cat(re_en_list).cpu()
    re_ex_list = torch.cat(re_ex_list).cpu()
    en_list=torch.cat(en_list)
    ex_list=torch.cat(ex_list)
    y_true=torch.cat(y_true)
    
    content_loss= torch.mean((re_en_list - en_list) ** 2, dim=1) # mse # only for evalute content loss

    # Initialize the MinMaxScaler
    scaler = MinMaxScaler()    
    content_loss=content_loss.reshape(-1,1) 
    content_loss = scaler.fit_transform(content_loss)
    _,_,content_auc,_= rocauc('beAnom',y_true, content_loss,plot=False) 
    
    context_loss = F.cross_entropy(re_ex_list.reshape(-1, re_ex_list.size(-1)), ex_list.reshape(-1), reduction='none')
    context_loss = context_loss.reshape(ex_list.size(0), ex_list.size(1))  # [seq_len, window_size]
    context_loss=context_loss.mean(dim=1)  
    
    context_loss=context_loss.reshape(-1,1) 
    context_loss = scaler.fit_transform(context_loss)    
    _,_,seq_auc,_= rocauc('beAnom',y_true, context_loss,plot=False) 
    
    loss= alpha* context_loss+ (1-alpha) *content_loss   
    scores = scaler.fit_transform(loss)  
    
    ################################  AUC-ROC   ########################################
    
    bestthresh,bestfpr,bestauc,_= rocauc('beAnom',y_true, scores,plot=True)  
    
    anom_label=(scores > bestthresh).astype(float)
            
    print(f"AUC-ROC: {bestauc:.4f}")
    print(f"\n Best Threshold = {bestthresh:.2f}")
    print(f"\n Best FPR = {bestfpr:.2f}")    
    
    report = precision_recall_fscore_support(y_true, anom_label,average='weighted')  

    print(f'Number of testing data: {str(len(test_loader))}')
    print(f'Precision: {report[0]:.4f}')
    print(f'Recall: {report[1]:.4f}')
    print(f'F1 score: {report[2]:.4f}') 

    ###################################  AUC-PR #################################################
    pr_threshold, pr_fpr, pr_auc, pr_f1=auc_pr_func(y_true, scores,plot=True)

    
    # Bundle metrics
    metrics_dic = {
        "datafile":test_filename,
        "modelname":'SD',
        "mode": args.mode,
        "epoch": epoch,
        "AUCROC": f"{bestauc:.4f}",
        "weighted_F1": f"{ report[2]:.4f}",
        "FPR": f"{bestfpr:.4f}",   # 0 is the best
        "precision":f"{report[0]:.4f}",
        "recall": f"{report[1]:.4f}",
        "threshold": f"{bestthresh:.2f}",
        "AUCROC_seq":f"{seq_auc:.4f}",
        "AUCROC_content":f"{content_auc:.4f}",

        "time_cost": time_cost,
        "SDparams":params_size,
    }
    log_metrics_to_csv(metrics_dic)
    
        
    if bestauc> final_auc:
        print(f"this hits the best AUC performance. AUC-ROC: {bestauc:.4f}, F1 score: {report[2]:.4f}")
        final_auc=bestauc
        torch.save(checkpoint, os.path.join(
            ckpt_path, f'train_bert_best-3.pt'))
        args.threshold_SD=bestthresh  

        #save y_true and scores for plot later# Save to CSV file
        output_filename="result/"+ test_filename.replace(".csv", "-yTruePred.csv")
          # Create a DataFrame
        results_df = pd.DataFrame({
            'y_true': y_true.flatten(),
            'y_pred': scores.flatten(),
        })

        # Save to CSV
        results_df.to_csv(output_filename, index=False)
        print(f"Results saved successfully to {output_filename}")


In [ ]:
del loss_all, train_loss #f1_all  train_pred_nm, train_true_nm, train_pred, train_true ,
del content_loss, context_loss, loss
del y_true #, y_pred, y_true_nm, y_pred_nm  # , y_pred_nm_b
del en_list, ex_list, re_en_list, re_ex_list
del recon_context,  recon_x, content_true, context_true
del train_generator, train_loader
del test_generator, test_loader
torch.cuda.empty_cache() 

In [ ]:
# y_true, anom_label
accuracy, recall = compute_metrics_recall(y_true, anom_label)
print(f"Sequence Detection Accuracy: {accuracy}")
print(f"Abnormal Event Recall (via context window): {recall}")


contextual_coverage = compute_metrics_coverrate(y_true, anom_label)
print(f"Contextual Abnormal Event Coverage: {contextual_coverage}")


## option: save the split index for recovery history if need

In [ ]:
save_args(args)


In [ ]:

combined_df = pd.concat([X_train_b['id'],X_test_b['id'],X_test['id']], axis=1)

combined_df.to_csv('result/historydataindex.csv', index=False)

# to do : load data according to the saved df

In [ ]:
args.save_sequences=None   # 'result/CB_seq_test'
# Save sequences if necessary
if args.save_sequences:
    with open(args.save_sequences, 'wb') as outfile:
        torch.save({
            "events" : events_test, # events,
            "context": context_test,  # context,
            "labels" : labels_test_binary,  # labels,
            "mapping": mapping,
        }, outfile)



# catch the footprint

In [ ]:
# load args params
args = load_args()


In [ ]:
args.load_builder="saved_model/saved_CB_model-3.pth"
context_builder = ContextBuilder.load(args.load_builder, args.device)



In [ ]:
transformermodel = Model(mode=args.mode, num_layers=args.num_layers_SD, dim=args.embed_content_dim, window_size=args.length, nhead=8, dim_feedforward=4 *
              args.embed_content_dim, dim_ff_reduced=128, dropout=0.3,eventID_size=args.events,eventID_embed_dim=args.embed_context_dim, 
                         param_vocab_size=args.inital_param_size, param_embed_dim=args.embed_param_dim,device=args.device)
transformermodel = transformermodel.to(device)   
transformermodel = torch.nn.DataParallel(transformermodel, device_ids=device_ids)

args.load_path='checkpoints/train_bert_best-3.pt'

path_checkpoint = args.load_path
checkpoint = torch.load(path_checkpoint)
filtered_dict = {k: v for k, v in checkpoint['net'].items() if k in transformermodel.state_dict()}
transformermodel.load_state_dict(filtered_dict, strict=False)


# label sequence using CB & bert model

## label whole file data

In [23]:
if args.inital_param_size ==0:
    f_v=data['field_values']
    unique_values = np.unique(f_v)  # Get unique values
    param_v_size=len(unique_values)
    print(param_v_size)

894054


In [19]:
event_template= pd.DataFrame( {'EventTemplate': data['field_names'].drop_duplicates(ignore_index=True)})


In [ ]:

args.dataloader_batch_size=250
args.dataloader_chunk_size=100000

transformermodel.eval()

chunk_size=args.dataloader_chunk_size

# Get the total size of the dataset
total_size = len(data)

# Calculate the number of chunks
num_chunks = (total_size + chunk_size - 1) // chunk_size

# Initialize list to store outputs
all_scores = []

# Process each chunk
for i in range(num_chunks):
    # Calculate start and end indices for this chunk
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, total_size)
    
    print(f"Processing chunk {i+1}/{num_chunks} (indices {start_idx} to {end_idx})")
    
    chunk_labels = labels_train_binary[start_idx:end_idx]
    chunk_context=context_train[start_idx:end_idx]
    chunk_events=events_train[start_idx:end_idx]


    _, attention_chunk = context_builder.predict(chunk_context,chunk_events.reshape(-1, 1))
    attention_chunk=attention_chunk.squeeze(1)
    
    fieldname_em_ta,fieldva_ind_ta=field_name_value_prepare(origdata_fieldname=data['field_names'][start_idx:end_idx], 
                                                        origdata_fieldvalue=data['field_values'][start_idx:end_idx],
                                                            embmodel=embmodel, event_template=event_template)
    attention_chunk=attention_chunk.to(fieldname_em_ta.device)
    chunk_context=chunk_context.to(fieldname_em_ta.device)
    
    # combination of field name, field value, attention vector 
    chunk_features=torch.cat([fieldname_em_ta,fieldva_ind_ta,attention_chunk, chunk_context],dim=1)



    # Create a DataGenerator and DataLoader for this chunk
    chunk_generator = DataGenerator(chunk_features, chunk_labels)
    chunk_loader = torch.utils.data.DataLoader(chunk_generator, batch_size=args.dataloader_batch_size, shuffle=False)
    
    # Process this chunk (collect outputs)        
    with torch.no_grad():  
        y_true=[]
        en_list, ex_list, re_en_list, re_ex_list=[],[],[],[]
        for batch_idx, (xx,y) in enumerate(tqdm(chunk_loader)): 
            content_true=xx[:,:args.embed_content_dim]
            context_true=xx[:,-args.length:].long()
            x=xx[:,:-args.length]
            x = x.to(device, dtype=torch.float32)
            # y = y.to(device, dtype=torch.float32)
            
            recon_content, recon_context = transformermodel(x)  
            
            y_true.append(y)
            en_list.append(content_true)
            ex_list.append(context_true)
            re_en_list.append(recon_content)
            re_ex_list.append(recon_context)
        
    re_en_list = torch.cat(re_en_list).cpu()
    re_ex_list = torch.cat(re_ex_list).cpu()
    
    en_list=torch.cat(en_list)
    ex_list=torch.cat(ex_list)
    y_true=torch.cat(y_true)
    content_loss = F.mse_loss(re_en_list, en_list)
    context_loss = F.cross_entropy(re_ex_list.reshape(-1, re_ex_list.size(-1)), ex_list.reshape(-1), reduction='none')
    loss= args.alpha_loss* context_loss+ (1-args.alpha_loss) *content_loss    
    loss = loss.reshape(ex_list.size(0), ex_list.size(1))  # [seq_len, window_size]
    seq_score=loss.mean(dim=1).reshape(-1,1).detach().numpy()
    # Initialize the MinMaxScaler
    scaler = MinMaxScaler()    
    chunk_score = scaler.fit_transform(seq_score)
    all_scores.append(chunk_score)
    
    # Clear memory
    del chunk_features, chunk_labels, chunk_generator, chunk_loader
    del fieldname_em_ta,fieldva_ind_ta,attention_chunk, chunk_context
    del y_true, en_list, ex_list, re_en_list, re_ex_list,
    # content_ture, context_true, recon_content, recon_context
    del loss, content_loss, context_loss, seq_score
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Combine all chunks
if all_scores:
    final_scores = np.concatenate(all_scores)


final_labels=(final_scores > args.threshold_SD).astype(float)

# 5 risk categories
final_levels=convert2level(final_scores, plot=True)
    

In [21]:
# Save to CSV file
output_filename= "result/"+ test_filename.replace(".csv", "-onlyScoreLabel.csv")
save_results_to_csv(final_scores, final_labels,final_levels, output_filename)

Saving results to result/18-19-052-onlyScoreLabel.csv
Results saved successfully to result/18-19-052-onlyScoreLabel.csv


In [22]:
# statistic
positive_ratio = (final_labels == 1).sum() / len(final_labels)
print(positive_ratio)

0.03998363529704134


## label training data for cluster

In [ ]:
if args.inital_param_size ==0:
    f_v=X_train['field_values']
    unique_values = np.unique(f_v)  # Get unique values
    param_v_size=len(unique_values)
    print(param_v_size)

    del unique_values,f_v

In [42]:
context_train=context_train.to(args.device)

In [ ]:
event_template= pd.DataFrame( {'EventTemplate': X_train['field_names'].drop_duplicates(ignore_index=True)})

########### to prepare  data
_, attention_ta = context_builder.predict(context_train,events_train.reshape(-1, 1))
attention_ta=attention_ta.squeeze(1)

fieldname_em_ta,fieldva_ind_ta=field_name_value_prepare(origdata_fieldname=X_train['field_names'], origdata_fieldvalue=X_train['field_values'], embmodel=embmodel, event_template=event_template)
attention_whole=attention_whole.to(fieldname_em_ta.device)
context_train=context_train.to(fieldname_em_ta.device)

# combination of field name, field value, attention vector 
ta_bert=torch.cat([fieldname_em_ta,fieldva_ind_ta,attention_ta, context_train],dim=1)
print(ta_bert.size())

In [ ]:
del f_v, unique_values, fieldname_em_ta,fieldva_ind_ta,attention_ta #, event_template, context_train
torch.cuda.empty_cache() 

In [ ]:


transformermodel.eval()
  
# ta_bert
train_seq_scores = process_large_data_SD(train_bert, labels_train_binary, model=transformermodel, args=args)

train_seq_labels=(train_seq_scores > args.threshold_SD).astype(float)

# 5 risk categories
train_seq_levels=convert2level(train_seq_scores, plot=True)

# Save to CSV file
output_filename="result/"+ test_filename.replace(".csv", "-onlyScoreLabel-3-train.csv")
save_results_to_csv(train_seq_scores, train_seq_labels, train_seq_levels, output_filename)
        


In [ ]:
# load CSV file
output_filename="result/"+ test_filename.replace(".csv", "-onlyScoreLabel-3-train.csv")
final_scores, final_labels, final_levels=load_anomaly_results(output_filename)

In [45]:
# statistic
positive_ratio = (train_seq_labels == 1).sum() / len(train_seq_labels)
# positive_ratio = (final_labels == 1).sum() / len(final_labels)
print(positive_ratio)

0.40288399343609893


## label testing data for evaluation

In [ ]:
if args.inital_param_size ==0:
    f_v=X_test['field_values']
    unique_values = np.unique(f_v)  # Get unique values
    param_v_size=len(unique_values)
    print(param_v_size)

    del f_v , unique_values

702769


In [ ]:
event_template= pd.DataFrame( {'EventTemplate': X_test['field_names'].drop_duplicates(ignore_index=True)})

########### to prepare  data
_, attention_t = context_builder.predict(context_test,events_test.reshape(-1, 1))
attention_t=attention_t.squeeze(1)

fieldname_em,fieldva_ind=field_name_value_prepare(origdata_fieldname=X_test['field_names'], origdata_fieldvalue=X_test['field_values'],  embmodel=embmodel, event_template=event_template)
attention_t=attention_t.to(fieldname_em.device)
context_test=context_test.to(fieldname_em.device)

# combination of field name, field value, attention vector 
te_bert=torch.cat([fieldname_em,fieldva_ind,attention_t,context_test],dim=1)
print(te_bert.size())

In [ ]:
del fieldname_em,fieldva_ind,attention_t # , event_template
torch.cuda.empty_cache() 

In [ ]:
''' due to large dataset, use DataLoader to split data, in case of out of memory'''
test_generator = DataGenerator( te_bert, labels_test_binary)
test_loader = torch.utils.data.DataLoader(
    test_generator, batch_size=1280, shuffle=False, num_workers=4)

<!-- 
1. data includes: training data, validate data, testing data
2. ContextBuilder, fit & predict on (training data), (validate data)
   TransformerModel, fit & predict on (training data), (validate data)
3.  ContextBuilder predict on testing data
    TransformerModel predict on testing data
4. based on trained ContextBuilder and TransformerModel, to produce the labels, 
[combine context sequences to labels,] work on testing data or new data.
5. Cluster in Interpreter
6. LLM
 -->

In [ ]:
transformermodel.eval()

y_true=[]
en_list, ex_list, re_en_list, re_ex_list=[],[],[],[]
for batch_idx, (xx,y) in enumerate(tqdm(test_loader)): 
    content_true=xx[:,:args.embed_content_dim]
    context_true=xx[:,-args.length:]
    x=xx[:,:-args.length]
    x = x.to(device, dtype=torch.float32)
    # y = y.to(device, dtype=torch.float32)

    recon_content, recon_context = transformermodel(x)  #.cpu()
    y_true.append(y)
    en_list.append(content_true)
    ex_list.append(context_true)
    re_en_list.append(recon_content)
    re_ex_list.append(recon_context)
        
re_en_list = torch.cat(re_en_list).cpu()
re_ex_list = torch.cat(re_ex_list).cpu()
en_list=torch.cat(en_list)
ex_list=torch.cat(ex_list)
y_true=torch.cat(y_true)

content_loss = F.mse_loss(re_en_list, en_list)
context_loss = F.cross_entropy(re_ex_list.reshape(-1, re_ex_list.size(-1)), ex_list.reshape(-1), reduction='none')
loss= alpha* context_loss+ (1-alpha) *content_loss    
loss = loss.reshape(ex_list.size(0), ex_list.size(1))  # [seq_len, window_size]
test_seq_scores=loss.mean(dim=1).reshape(-1,1)
# Initialize the MinMaxScaler
scaler = MinMaxScaler()    
test_seq_scores = scaler.fit_transform(test_seq_scores)

test_seq_labels=(test_seq_scores > args.threshold_SD).astype(float)

labels_test_binary=labels_test_binary


# 5 risk categories
train_seq_levels=convert2level(train_seq_scores, plot=True)


# Save to CSV file
output_filename="result/"+ test_filename.replace(".csv", "-onlyScoreLabel-test.csv")
save_results_to_csv(train_seq_scores, train_seq_labels, train_seq_levels, output_filename)


In [ ]:
del y_true, en_list, ex_list, re_en_list, re_ex_list
del recon_content, recon_context, content_loss, context_loss, loss,scaler
del test_generator, test_loader
torch.cuda.empty_cache() 

In [ ]:
# Convert to NumPy
scores_np = test_seq_scores.detach().cpu().numpy()

# --- Plot ---
plt.figure(figsize=(10, 4))
sns.lineplot(x=np.arange(len(scores_np)), y=scores_np, marker="o")
plt.title("Anomaly Scores Per Log Sequence (Autoencoder Reconstruction)")
plt.xlabel("Sequence Index")
plt.ylabel("Reconstruction Loss (Anomaly Score)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [54]:
'''Recall = (# of abnormal items that are covered by at least one prediction in their forward context) / (total # of abnormal items)
his version measures how many of the real abnormal events are caught by at least one predicted label in the relevant prediction window.
This is a recall-style metric for forward-looking predictions, adapted to sequence data.
'''

def compute_metrics_recall(sequence_item_labels, predicted_sequence_labels, window_size=10):
    assert len(sequence_item_labels) == len(predicted_sequence_labels)

    # Compute accuracy
    correct = sum(p == t for p, t in zip(predicted_sequence_labels, sequence_item_labels))
    accuracy = correct / len(sequence_item_labels)

    # Compute contextual recall
    total_abnormal = sum(1 for label in sequence_item_labels if label == 1)
    detected_abnormal = 0

    for i, label in enumerate(sequence_item_labels):
        if label == 1:
            # Check if this abnormal item is within the forward context of any prediction
            start = max(0, i - window_size + 1)
            if any(predicted_sequence_labels[j] == 1 for j in range(start, i + 1)):
                detected_abnormal += 1

    recall = detected_abnormal / total_abnormal if total_abnormal > 0 else 0.0

    return accuracy, recall

''' how many predicted positives (1s) actually caught at least one abnormal event (1) in the forward window.
Reconstruct sequence_item_labels into a context matrix, where each row is a forward context window of size 10.
For each position i in predicted_sequence_labels where the prediction equals 1:
Look at the corresponding context window from the matrix.
Check if the window contains at least one 1.
Count how many of those predictions (==1) actually cover at least one abnormal item.
'''
def compute_metrics_coverrate(sequence_item_labels, predicted_sequence_labels, window_size=10):
    
    # Build forward context matrix from sequence_item_labels
    context_matrix = []
    for i in range(len(sequence_item_labels)):
        window = sequence_item_labels[max(0,i-window_size) :i]
        # Pad with zeros if the window goes out of bounds
        if len(window) < window_size:
            window = window + [0] * (window_size - len(window))
        context_matrix.append(window)

    # Evaluate predicted=1 windows for abnormal event presence
    abnormal_predictions = 0
    correct_abnormal_predictions = 0
    for pred, context in zip(predicted_sequence_labels, context_matrix):
        if pred == 1:
            abnormal_predictions += 1
            if 1 in context:
                correct_abnormal_predictions += 1

    # Coverage-like metric: precision over predicted=1 windows that contain abnormal events
    contextual_coverage = (
        correct_abnormal_predictions / abnormal_predictions
        if abnormal_predictions > 0 else 0.0
    )

    return contextual_coverage
    



In [ ]:

accuracy, recall = compute_metrics_recall(labels_test_binary, test_seq_labels)
print(f"Sequence Detection Accuracy: {accuracy:.2f}")
print(f"Abnormal Event Recall (via context window): {recall:.2f}")


contextual_coverage = compute_metrics_coverrate(labels_test_binary, test_seq_labels)
print(f"Contextual Abnormal Event Coverage: {contextual_coverage:.2f}")


# Interpreter
cluster on training data.
predict the score of cluster which the most similar sequence in. otherwise the predict label is -2, means no label data in training.
result : np.array of shape=(n_samples,)
                Predicted maliciousness score.
                Positive scores are maliciousness scores.
                A score of 0 means we found a match that was not malicious.
                Special cases:

                * -1: Not confident enough for prediction
                * -2: Label not in training
                * -3: Closest cluster > epsilon

In [17]:
# option1 Load results from CSV
label_filename="result/"+ test_filename.replace(".csv", "-onlyScoreLabel.csv")
final_scores, final_labels, final_levels = load_anomaly_results(label_filename)


Loading anomaly results from result/AIA-201-225.ecar-onlyScoreLabel.csv
Successfully loaded 23189497 records
Anomaly distribution: 7538211.0 anomalies, 15651286.0 normal


In [ ]:
''' adjust the hypterparameter of Interepreter'''
args.epsilon=0.1 
# # args.min_samples=5
args.confidence=0.1

print( args.epsilon)
print( args.min_samples)
print( args.confidence)

In [ ]:
########################################################################
#                             Interpreter                            #
########################################################################
# args.load_interpreter="saved_Interpreter_model-2" # 
args.load_interpreter=None   # args.save_interpreter

# Load the interpreter, if necessary
if args.load_interpreter:
    interpreter = Interpreter.load(
        args.load_interpreter,
        context_builder = context_builder,
    )
# Otherwise create a new Interpreter
else:
    # Create Interpreter
    interpreter = Interpreter(
        context_builder = context_builder,
        features        = args.events,
        eps             = args.epsilon,
        min_samples     = args.min_samples,
        threshold       = args.confidence,
    )



In [ ]:
events_train  = events_train.to(args.device)
context_train  = context_train.to(args.device)

## cluster in detail

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

def sample_fullcluster( vector_set, context_mask,ration=0.01):
# for the too large vecs to cluster:
    sample_ratio = ration  # e.g., 1%
    sampled_indices = []
    
    # 1. choose sample
    # for _, context_mask in indices_y[:1]:
    indices = context_mask.nonzero(as_tuple=True)[0] if context_mask.dtype == torch.bool else context_mask
    n_sample = max(1, int(len(indices) * sample_ratio))
    sampled_indices = indices[torch.randperm(len(indices))[:n_sample]]
    # sampled_indices.append(sampled)
    
    # sampled_indices = torch.cat(sampled_indices)
    sampled_vectors = vector_set[sampled_indices]  #.cpu().numpy()

    #2. cluster samples
    sampled_labels = interpreter.dbscan.dbscan(
        X=sampled_vectors,
        eps=interpreter.eps,
        min_samples=interpreter.min_samples,
        verbose=True,
    )
    # Compute cluster centroids
    valid = sampled_labels != -1
    sampled_centroids = []
    sampled_cluster_ids = []
    
    for label in np.unique(sampled_labels[valid]):
        mask = (sampled_labels == label)
        centroid = sampled_vectors[mask].mean(axis=0)
        sampled_centroids.append(centroid)
        sampled_cluster_ids.append(label)
    
    sampled_centroids = np.vstack(sampled_centroids)  # shape: (k, dim)

    sampled_centroids = np.asarray(sampled_centroids)


# for _, context_mask in indices_y[:1]:
    indices = context_mask.nonzero(as_tuple=True)[0] if context_mask.dtype == torch.bool else context_mask
    vecs = vector_set[context_mask]  #.cpu().numpy()

    # Compute cosine distances to centroids
    distances = cosine_distances(vecs, sampled_centroids)
    nearest = np.argmin(distances, axis=1)

    # Assign labels based on nearest centroid
    labels = np.array([sampled_cluster_ids[i] for i in nearest])
    # result[indices] = torch.from_numpy(labels)

    return labels
    

In [ ]:
from deepcase.interpreter.utils import  group_by

xx=context_train
yy= events_train.reshape(-1, 1)

In [ ]:
have_vector==False

if have_vector==True:
    vectors=torch.load("checkpoints/interpreter_vectors_all.pt ")
    mask=torch.load("checkpoints/interpreter_mask_all.pt ")

else:
    # Get optimized vectors
    vectors, mask = interpreter.attended_context(
        X                =xx ,
        y                =yy,
        threshold        = interpreter.threshold ,
        iterations       = 3, # 100,
        batch_size       = 2048,
        verbose          = True, #verbose,
    )
    
    torch.save( vectors, "checkpoints/interpreter_vectors_all.pt")
    torch.save( mask, "checkpoints/interpreter_mask_all.pt")



####################################################################
#                     Group sequences by event                     #
####################################################################

# Group sequences for clustering per event type
indices_y = group_by(
    X   = yy[mask].squeeze(1).cpu().numpy(),
    key = lambda x: x.data.tobytes(),
)

# Add verbosity if necessary
indices_y = tqdm(indices_y, desc="Clustering")

####################################################################
#                          Cluster events                          #
####################################################################

# Initialise result for confident samples
result = np.full(mask.sum(), -1, dtype=int)

#save temp cluster in disk files
os.makedirs("temp_clusters", exist_ok=True)

offset = 0

for idx, (event, context_mask) in enumerate(indices_y):
    if vectors[context_mask].shape[0]>1000000:
        clusters=sample_fullcluster(vectors,context_mask)
        print("large event group:")
        print(vectors[context_mask].shape[0])
    else:
        # Run DBSCAN
        clusters = interpreter.dbscan.dbscan(
            X=vectors[context_mask],
            eps=interpreter.eps,
            min_samples=interpreter.min_samples,
            verbose=True,
        )

    # Adjust cluster labels to be globally unique
    clusters[clusters != -1] += offset

    # Save clusters to disk
    np.save(f"temp_clusters/clusters_{idx}.npy", clusters)

    # Update offset
    if clusters[clusters != -1].size > 0:
        offset = clusters.max() + 1

    print(idx)


# load to result list
# Initialise result for confident samples
for idx, (event, context_mask) in enumerate(indices_y):
    clusters = torch.from_numpy(np.load(f"temp_clusters/clusters_{idx}.npy"))
    result[context_mask] = clusters


In [ ]:
####################################################################
#                    Add non-confident clusters                    #
####################################################################

# Set clusters to -1 by default, i.e., if not confident
clusters = np.full(mask.shape[0], -1, dtype=int)
# Add confident clusters
clusters[mask.cpu().numpy()] = result

####################################################################
#                         Store in object                          #
####################################################################

# Store clusters
interpreter.clusters = clusters
# Store vectors
interpreter.vectors = vectors
# Store events
interpreter.events = yy.reshape(-1).cpu().numpy()


import shutil

shutil.rmtree("temp_clusters")

## cluster in function

In [ ]:
# Cluster samples with the interpreter
clusters ,confindence_mask= interpreter.cluster(
    X          = context_train,  # context,    #
    y          = events_train.reshape(-1, 1),   #  events.reshape(-1, 1),  #
    iterations = 5,  # 100,
    batch_size = 1024,
    verbose    = args.silent,  # not args.silent,
)
# Save clusters, if necessary
if args.save_clusters:

    # Set labels to -1 if no labels were provided
    if seq_label is None:
        seq_label = np.full(clusters.shape[0], -1, dtype=int)

    # Save to file
    pd.DataFrame({
        'clusters': clusters,
        'labels'  : seq_label,
    }).to_csv(args.save_clusters, index=False)

## score

In [ ]:
# # in case of final_scores.dim>1
if len(final_scores.shape)>1:
    final_scores=final_scores[:,0]
    
if len(final_labels.shape)>1:
    final_labels=final_labels[:,0]    
    
print(final_scores.shape)
print(final_labels.shape)

In [ ]:
# Compute scores for each cluster based on individual labels/score/level per sequence
args.cluster_metric= "labels"  # "levels" # 

if args.cluster_metric=="scores":
    seq_metric= final_scores  # train_seq_scores  
elif args.cluster_metric=="labels":
    seq_metric= final_labels  #  train_seq_labels
elif args.cluster_metric=="levels":
    seq_metric= final_levels #  train_seq_levels

    
scores = interpreter.score_clusters(
    scores   = seq_metric, #    # metric( e.g.Labels) used to compute score (either as loaded by Preprocessor, or put your own labels here)               
    strategy = "avg",        # Strategy to use for scoring (one of "max", "min", "avg")
    NO_SCORE = -1,           # Any sequence with this score will be ignored in the strategy.
                                # If assigned a cluster, the sequence will inherit the cluster score.
                                # If the sequence is not present in a cluster, it will receive a score of NO_SCORE.
)

In [ ]:
# Assign scores to clusters in interpreter
# Note that all sequences should be given a score and each sequence in the
# same cluster should have the same score.
interpreter.score(
    scores  = scores, # Scores to assign to sequences
    verbose = True,   # If True, prints progress
)


# Save the interpreter, if necessary
# args.save_interpreter=None
args.save_interpreter="saved_model/saved_Interpreter_"+test_filename
if args.save_interpreter:
    interpreter.save(args.save_interpreter)
    

## evaluate Interpreter-manual

### the distribute of clusters

In [ ]:
#  plot cluster items distribution
unique_values, counts = np.unique(clusters, return_counts=True)

print("Unique values:", unique_values)
print("Counts:", counts)

# Define custom bins
bin_edges = [1000, 10000, 50000, 100000, np.inf]

# Compute histogram manually (if you want counts)
hist, edges = np.histogram(counts, bins=bin_edges)

# Optional: custom x-axis labels
labels = ["1K-10K","10K-50K","50K-100K", "100K+"]

# Plot histogram
plt.figure(figsize=(8, 5))
plt.bar(labels, hist, width=0.6, edgecolor='black')
plt.title('Histogram of Cluster Sizes')
plt.xlabel('Cluster Size (number of items)')
plt.ylabel('Number of Clusters')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(counts, bins=range(1, 1001), edgecolor='black', align='left')  # max(counts)+2
plt.title('Histogram of Cluster Sizes')
plt.xlabel('Cluster Size (number of items)')
plt.ylabel('Number of Clusters')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### metrics on manul analysis

In [ ]:
''' reduced_FP_train= all clustered event seq (suspicous+ non-suspicous)/all event seq  ;
next step need to manual set/label suspicous clusters '''
unexpected_seq = np.isin(clusters, [-1, -2, -3]).sum()
print("Count of -1, -2, or -3:", unexpected_seq)
reduced_FP_train=1- ( unexpected_seq/len(clusters))
print(f"reduced FP on train set/manual analysis is {reduced_FP_train}")

'''coverage = event seq > confidence_threshold
'''
# unconfidence_seq = np.isin(clusters, [-1]).sum()
# coverage= (len(clusters)-unconfidence_seq)  / len(clusters)
unconfidence_seq = len( confidence_mask[np.where(confidence_mask==True)] ) # 1=True, 0=False
coverage= unconfidence_seq / len(clusters)
print(f"coverage on train set/manual analysis is {coverage}, that means confidence rate of whole data.")


'''suspicous cluster rate= suspicous clusters/all clusters , all clusters removed unexpected 3 situations.
'''
mask = ~np.isin(unique_values, [-1, -2, -3])

count_suspicious_cluster=0
# Group scores by cluster index
# for cluster_id, score in zip(clusters, scores):
for cluster_id in unique_values[mask]:
    temp_a=scores[clusters==cluster_id].mean()
    if isinstance(temp_a, float) and not temp_a.is_integer():
        count_suspicious_cluster=count_suspicious_cluster+1

suspi_cluster_rate=count_suspicious_cluster/len(unique_values[mask])
print(f"suspicous clusters rate on train set/manual analysis is {suspi_cluster_rate}")


''' suspicous rate= suspicous event seq /all event seq that remove 3 unexpected event seqs.'''
# Count non-integer floats
non_integer_floats = [x for x in scores if isinstance(x, float) and not x.is_integer()]
suspi_rate=len(non_integer_floats)/ ( len(clusters)-unexpected_seq )
print(f"suspicious rate on train set/manual analysis is {suspi_rate}")


In [ ]:
# Bundle metrics
metrics_dic = { 
    "datafile":test_filename,
    "modelname":'Interpreter',
    "mode": args.mode,
    "seq_metrics":args.cluster_metric,
    "AUCROC":  '0',
    "weighted_F1": '0',
    "accuacy": '0',    
    "FPR": '0',  
    "precision":'0',
    "recall": '0',
    "threshold": '0',
    "confidence_threshold":f"{args.confidence:.4f}",
    "eps":f"{args.epsilon:.4f}",
    "reduced_FP_train":f"{reduced_FP_train:.2f}",
    "coverage":f"{coverage:.2f}",
    "suspi_cluster_rate":f"{suspi_cluster_rate:.2f}",
    "suspi_rate":f"{suspi_rate:.2f}",
    "reduce_FP_test":'0',
    "susp_rate_test":'0',
    "susp_rate_expected":'0',
    "coverage_test":'0',
    "susp_cluster_rate":'0',
}
log_metrics_to_csv(metrics=metrics_dic,csv_path="result/Interp_test_metrics.csv") 





## predict

In [ ]:
########################################################################
#                       Semi-automatic analysis                      #
########################################################################
    
#Compute predicted scores
if len(events_test)>args.dataloader_chunk_size * 3:
    pred, indexIntraining= process_large_data_Interp_predict(context_test, events_test.reshape(-1, 1), args, interpreter)
else:
    pred, indexIntraining = interpreter.predict(
        X          = context_test,               # Context to predict
        y          = events_test.reshape(-1, 1), # Events to predict, note that these should be of shape=(n_events, 1)
        iterations = 3, # 10,  # 100,                        # Number of iterations to use for attention query, in paper this was 100
        batch_size = 1024,                       # Batch size to use for attention query, used to limit CUDA memory usage
        verbose    = True,                       # If True, prints progress
    )


# # Save to file
# pd.DataFrame({
#     'labels': prediction,
# }).to_csv(args.save_prediction, index=False)


In [ ]:

unique_vals = np.unique(pred)
# 
if args.cluster_metric=="scores":
    # when using seqence final scores
    mask_metric= np.where((pred !=-1) & (pred !=-2) & (pred !=-3))[0]

    # option 1, using search method to find best threshold
    # m_score, m_threshold=threshold_search(labels_test_binary[mask_metric], pred[mask_metric])
    # print(f"\n Best Threshold by threshold search = {m_threshold:.2f}")
    # print(f"Accuracy    = {m_score:.4f}")
    # prediction=pred
    # prediction[mask_metric]=(pred[mask_metric] > m_threshold).astype(float)

    # option2 , using rocauc to get best threshold
    bestthresh,bestfpr,bestauc,_= rocauc('beAnom',labels_test_binary[mask_metric], pred[mask_metric],plot=True)      
    prediction=pred
    prediction[mask_metric]=(pred[mask_metric] > bestthresh).astype(float)        
    print(f"prediction2   AUC-ROC: {bestauc:.4f}")
    print(f"\n prediction2   Best Threshold = {bestthresh:.2f}")
    print(f"\n  prediction2    Best FPR = {bestfpr:.2f}")  
    print(classification_report(labels_test_binary,prediction,digits=4))
  
    test_labels=labels_test_binary
    
elif args.cluster_metric=="labels":
    # when using sequence final labels
    mask_metric= [ i for i, x in enumerate(pred)
    if (isinstance(x, int) or (isinstance(x, float) and x.is_integer()))
                 ]
    mask_metric_suspicous=np.setdiff1d(np.arange(len(pred)), np.array(mask_metric), assume_unique=True)
    prediction=pred[mask_metric]
    labels_test=labels_test_binary[mask_metric]
    
elif args.cluster_metric=="levels":
# when using sequence final levels
    mask_metric = [ i for i, x in enumerate(pred)
    if (isinstance(x, int) or (isinstance(x, float) and x.is_integer()))
                 ]    
    mask_metric_suspicous=np.setdiff1d(np.arange(len(pred)), np.array(mask_metric), assume_unique=True)
    prediction=pred[mask_metric]
    labels_test=labels_test_binary[mask_metric]

else:
    print("can't recognise prediction content.")
    
print("Next evaluate the normal and anomalous, and three unexpected scenes:")
# the mask_metric, has removed suspicious clusters.
# If labels were provided, print classification report
if labels_test is not None:
    print("Classification report")
    print(classification_report(
        y_pred        = prediction,
        y_true        = labels_test,
        digits        = 4,
        zero_division = 0,
    ))

    # Print confusion matrix
    print("Confusion matrix")
    all_labels = np.unique(labels_test).tolist()
    print(confusion_report(
        y_pred       = prediction,
        y_true       = labels_test,
        labels       = [-3, -2, -1] + all_labels,
        target_names = ['LOW EPS', 'NOT IN TRAIN','LOW CONFIDENCE' ] + all_labels,
        skip_x       = all_labels,
        skip_y       = ['LOW EPS', 'NOT IN TRAIN','LOW CONFIDENCE']
    ))

#Compute the accuracy, remove items in [-1, -2, -3]
print("Next only evaluate the normal and anomalous:") 
mask_p_n = np.where((prediction !=-1) & (prediction !=-2) & (prediction !=-3))[0]
result_predicted = prediction[mask_p_n]
seq_label_mask = labels_test[mask_p_n]

report_class=classification_report(seq_label_mask,result_predicted,digits=4,output_dict=True)
print(report_class)
print(f"weighted F1: {report_class['weighted avg']['f1-score']}") # Overall accuracy, real-world data with imbalance

In [ ]:
''' filter 
'''
# the reduced false positive rate 
reduce_FP_rate=(len(result_predicted) +len( mask_metric_suspicous) )     /len(pred) # where the items can make a clarify prediction.
print(f"the reduced false positive rate is {reduce_FP_rate}. this means these percent items can be predicted by the model into normal or abnormal.")

#the percent of event in suspicious clusters in whole data (1)/ or whole clustered records (2).
susp_rate_whole=len(mask_metric_suspicous)/len(pred) # (1)
susp_rate_clustered=len(mask_metric_suspicous)/ ( len(mask_metric_suspicous) + len(result_predicted)) # (2)

print(f"the percent of suspicious events in whole dataset, {susp_rate_whole}") # 
print(f"the percent of suspicious events in clustered dataset, {susp_rate_clustered}")

# coverage rate on test data
unconfidence_seq_test=len( pred[np.where (pred==-1 )]  )
coverage_test=1- ( unconfidence_seq_test/len(pred))

print(f"the coverage in test dataset, {coverage_test}")

# # suspicous cluster rate based on expeceted clusters, remove -1,-2,-3
# susp_cluster_rate=?

In [ ]:
# Bundle metrics
metrics_dic = { 
    "datafile":test_filename,
    "modelname":'Interpreter',
    "mode": args.mode,
    "seq_metrics":args.cluster_metric,
    "AUCROC":  f"{bestauc:.4f}" if args.cluster_metric=="scores" else '0',
    "weighted_F1": f"{ report_class['weighted avg']['f1-score']:.4f}",
    "accuacy": f"{report_class['accuracy']:.4f}",    
    "FPR": f"{bestfpr:.4f}" if args.cluster_metric=="scores" else '-1',  # 0 is the best
    # "precision":f"{report[0]:.4f}",
    # "recall": f"{report[1]:.4f}",
    "threshold": f"{bestthresh:.2f}" if args.cluster_metric=="scores" else '0',
    "confidence_threshold":f"{args.confidence:.4f}",
    "eps":f"{args.epsilon:.4f}",
    "reduced_FP_train":'0',
    "coverage":'0',
    "suspi_cluster_rate":'0',
    "suspi_rate":'0',
    "reduce_FP_test":f"{reduce_FP_rate:.2f}",
    "susp_rate_test":f"{susp_rate_whole:.2f}",
    "susp_rate_expected":f"{susp_rate_clustered:.2f}",
    "coverage_test":f"{coverage_test:.2f}",
    # "susp_cluster_rate":f"{susp_cluster_rate:.2f}",
}
log_metrics_to_csv(metrics=metrics_dic,csv_path="result/Interp_test_metrics.csv")


In [ ]:
'''find out the origin record, with label, score, clusers for similar seqs'''
print( X_train.iloc[indexIntraining].iloc[0,:] )